# PCA analyses for ancestry filtering

## 1000 Genomes whole cohort PCA
Round 1: whole-genome PCA on all 1000G populations, HapMap3 SNPs, filtered on MAF, HWE, missingness, LD.

## Setup

plink2: manual install, not conda. Bucket: already mounted at `~/workspace/` by Verily Workbench — no manual `gcsfuse` step needed.

In [ ]:
%%bash
set -e

# plink2
BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  # URL is dated; if it 404s, get current link from https://www.cog-genomics.org/plink/2.0/
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

export PATH="$BIN_DIR:$PATH"
plink2 --version

ls "$HOME/workspace"

In [ ]:
import os

# %%bash cells don't inherit PATH changes from prior cells — set it here instead
bin_dir = os.path.expanduser("~/bin")
if bin_dir not in os.environ["PATH"].split(":"):
    os.environ["PATH"] = f"{bin_dir}:{os.environ['PATH']}"

## Paths

`BUCKET_DIR`: under `~/workspace/` (auto-mounted by Verily Workbench). `BUCKET_RESOURCE_NAME` below is a placeholder — set it from the `ls` output in the Setup cell. See README's Layout convention for the full bucket structure.

In [ ]:
import os

# os.path.expanduser resolves ~ here, in Python — never pass a literal "~" into
# a bash variable via %%bash -s, since bash only expands ~ in literal command
# text, not in the *value* of a substituted variable ("$1", "$VAR", etc.)
WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)

MOUNT_DIR = WORKSPACE_BUCKET
BUCKET_DIR = f"{MOUNT_DIR}/data/01_ancestry_filtering"
os.makedirs(BUCKET_DIR, exist_ok=True)

BFILE = f"{BUCKET_DIR}/1kg_all_chrs"                # genome-wide bed/bim/fam prefix
HM3_SNPLIST = f"{BUCKET_DIR}/hm3.snplist"
PANEL = f"{BUCKET_DIR}/integrated_call_samples_v3.20130502.ALL.panel"

OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_hm3"            # post-QC bed/bim/fam
PRUNE_PREFIX = f"{BUCKET_DIR}/all_pruned"
PCA_PREFIX = f"{BUCKET_DIR}/round1_pca/1kg_all_pca"
os.makedirs(f"{BUCKET_DIR}/round1_pca", exist_ok=True)

print(BUCKET_DIR)

## Build reference files

Prebuilt 1000G plink files ([plink2 resources](https://www.cog-genomics.org/plink/2.0/resources): 3,202 samples, GRCh38, pgen/pvar/psam, "noannot" pvar variant) — no VCF download/merge needed. Restricted to HapMap3 SNPs (no-MHC list, hosted by the Broad's alkesgroup bucket — the old `data.broadinstitute.org` bz2 link is dead) and the classic 2,504 unrelated samples (via `PANEL`, same file round 2 uses) to stay consistent with round 2's ellipsoid fit.

In [ ]:
%%bash -s "$BUCKET_DIR" "$BFILE" "$HM3_SNPLIST" "$PANEL"
set -e
BUCKET_DIR=$1
BFILE_PATH=$2
HM3_SNPLIST_PATH=$3
PANEL_PATH=$4

cd "$BUCKET_DIR"

BFILE_NAME=$(basename "$BFILE_PATH")
HM3_NAME=$(basename "$HM3_SNPLIST_PATH")
PANEL_NAME=$(basename "$PANEL_PATH")

command -v zstd >/dev/null 2>&1 || {
  echo "zstd not found — install it first (e.g. 'sudo apt-get install -y zstd')" >&2
  exit 1
}

# -s (not -f): treat a 0-byte file as missing, not already-fetched. A failed
# download can still leave an empty file behind (e.g. `cmd > dest` creates dest
# before cmd runs, even if cmd then fails) — that bit us here.
fetch() {
  local dest=$1 url=$2
  [ -s "$dest" ] && return 0
  curl -fL --retry 5 --retry-delay 5 --retry-all-errors -o "${dest}.part" "$url"
  mv "${dest}.part" "$dest"
}

# Large .zst files from this host are prone to silent truncation mid-transfer.
# zstd -t alone isn't sufficient to catch it: these files are multi-frame, and a
# transfer cut off exactly at a frame boundary still passes zstd -t even though
# the overall file is incomplete. So this checks the downloaded byte count against
# the server's real Content-Length first, then zstd -t as a secondary check.
fetch_zst() {
  local dest=$1 url=$2
  [ -s "$dest" ] && return 0
  local want
  want=$(curl -sIL "$url" | awk 'BEGIN{IGNORECASE=1} /^content-length:/{v=$2} END{gsub("\r","",v); print v}')
  for attempt in 1 2 3; do
    curl -fL --retry 5 --retry-delay 5 --retry-all-errors -o "${dest}.part" "$url"
    local got
    got=$(wc -c < "${dest}.part")
    if [ -n "$want" ] && [ "$got" != "$want" ]; then
      echo "Size mismatch for $dest (got $got, expected $want) on attempt $attempt, retrying" >&2
      rm -f "${dest}.part"
      continue
    fi
    if zstd -t "${dest}.part" 2>/dev/null; then
      mv "${dest}.part" "$dest"
      return 0
    fi
    echo "zstd integrity check failed on attempt $attempt for $dest, retrying" >&2
    rm -f "${dest}.part"
  done
  echo "Failed to download a valid $dest after 3 attempts" >&2
  exit 1
}

# HapMap3 SNP list (no-MHC), plain text, no header
fetch "$HM3_NAME" "https://storage.googleapis.com/broad-alkesgroup-public/Variant_effects/1000G_EUR_Phase3_hg38/w_hm3.noMHC.snplist"

# classic 2,504 sample panel
fetch "$PANEL_NAME" "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/integrated_call_samples_v3.20130502.ALL.panel"
awk 'NR>1 {print $1}' "$PANEL_NAME" > classic_2504.keep

# prebuilt 1000G plink files (3,202 samples, GRCh38, 2022-08-04 Byrska-Bishop build).
# these are Dropbox "scl/fi" share links, which require both the file id AND an
# rlkey query param — a link missing rlkey resolves to an HTML page, not the file.
# pvar is the "noannot" (rsID-only, no INFO fields) build: much smaller (556 MiB vs
# 2.68 GiB) and confirmed to have the same 75,193,455-variant count as the pgen —
# the "_rs.pvar" (annotated) version's Dropbox link was serving a truncated file.
# psam/pvar fetched directly under the resources page's recommended "all_hg38.*"
# names rather than their as-downloaded names.
if [ ! -f all_hg38.pgen ]; then
  fetch_zst all_hg38.pgen.zst "https://www.dropbox.com/s/j72j6uciq5zuzii/all_hg38.pgen.zst?dl=1"
  zstd -d --rm all_hg38.pgen.zst
fi
fetch_zst all_hg38.pvar.zst "https://www.dropbox.com/scl/fi/id642dpdd858uy41og8qi/all_hg38_rs_noannot.pvar.zst?rlkey=sskyiyam1bsqweujjmxqv1h55&dl=1"
fetch all_hg38.psam "https://www.dropbox.com/scl/fi/u5udzzaibgyvxzfnjcvjc/hg38_corrected.psam?rlkey=oecjnk4vmbhc8b1p202l0ih4x&dl=1"

if [ ! -f "${BFILE_NAME}.bed" ]; then
  # restrict to classic samples + HM3 SNPs by native rsID
  # --pvar takes the full filename (incl. .zst) directly, no 'vzs' modifier needed —
  # 'vzs' is only for --pfile, which takes a bare prefix
  plink2 \
    --pgen all_hg38.pgen \
    --pvar all_hg38.pvar.zst \
    --psam all_hg38.psam \
    --keep classic_2504.keep \
    --max-alleles 2 \
    --rm-dup exclude-all \
    --extract "$HM3_NAME" \
    --make-bed \
    --out 1kg_hm3_raw

  # relabel IDs to chr:pos:ref:alt for matching against ACAF later
  plink2 \
    --bfile 1kg_hm3_raw \
    --set-all-var-ids '@:#:$r:$a' \
    --new-id-max-allele-len 1000 \
    --make-bed \
    --out "$BFILE_NAME"

  rm -f 1kg_hm3_raw.{bed,bim,fam,log}
fi

# all_hg38.pgen decompresses to ~60 GiB (75.2M variants x 3,202 samples); once
# BFILE is built (HM3-restricted, ~340 MiB) the raw genome-wide files are never
# read again, so reclaim the space rather than leaving them in the bucket
rm -f all_hg38.pgen all_hg38.pvar.zst

wc -l "${BFILE_NAME}.bim" "$HM3_NAME" "${BFILE_NAME}.fam" classic_2504.keep

## Variant QC + LD pruning

No `--keep` (unlike round 2) — all populations, already HM3-restricted.

In [ ]:
%%bash -s "$BFILE" "$OUT_PREFIX" "$PRUNE_PREFIX"
BFILE=$1
OUT_PREFIX=$2
PRUNE_PREFIX=$3

plink2 \
  --bfile "$BFILE" \
  --maf 0.01 \
  --hwe 1e-5 \
  --geno 0.05 \
  --max-alleles 2 \
  --rm-dup exclude-all \
  --make-bed \
  --out "$OUT_PREFIX"

plink2 \
  --bfile "$OUT_PREFIX" \
  --indep-pairwise 200kb 1 0.1 \
  --out "$PRUNE_PREFIX"

wc -l "${PRUNE_PREFIX}.prune.in"

## PCA

20 PCs, `allele-wts` — loadings reused below to project AoU onto this space.

In [ ]:
%%bash -s "$OUT_PREFIX" "$PRUNE_PREFIX" "$PCA_PREFIX"
OUT_PREFIX=$1
PRUNE_PREFIX=$2
PCA_PREFIX=$3

plink2 \
  --bfile "$OUT_PREFIX" \
  --extract "${PRUNE_PREFIX}.prune.in" \
  --freq counts \
  --pca allele-wts 20 \
  --out "$PCA_PREFIX"

ls -lh "${PCA_PREFIX}".*

## Project AoU data onto the 1000G PCs

**File/format:** AoU ACAF threshold callset (GRCh38 srWGS, AF-filtered, chromosome-split pgen/pvar/psam).

**Variant filtering:** intersect, not re-filter — `--extract` the reference's already-pruned SNP list (`${PRUNE_PREFIX}.prune.in`) after re-IDing ACAF to the same `chr:pos:ref:alt` scheme. Overlap % reported as a sanity check.

**Mechanics:** `plink2 --score` against `.eigenvec.allele` + `.acount` from the PCA cell.

**Unconfirmed:** exact env var for the ACAF path (varies by CDR version). Next cell scans `os.environ` for it.

In [ ]:
import os

{k: v for k, v in os.environ.items() if "ACAF" in k.upper()}

In [ ]:
# AOU_ROOT: parent of the shared-env-pilot dir set in WORKSPACE_BUCKET above —
# PLINK_BED lives under a sibling directory, not under shared-env-pilot itself
AOU_ROOT = os.path.dirname(WORKSPACE_BUCKET)
assert os.path.isdir(AOU_ROOT), f"AOU_ROOT does not exist: {AOU_ROOT!r}"

# native pgen export, not plink_bed (PLINK1 bed) — scattered/pruned-SNP extraction
# against plink_bed over the network-mounted filesystem hangs; pgen doesn't
ACAF_PGEN_DIR = f"{AOU_ROOT}/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/pgen"
assert os.path.isdir(ACAF_PGEN_DIR), f"ACAF_PGEN_DIR does not exist: {ACAF_PGEN_DIR!r}"

ACAF_PGEN_PATTERN = ACAF_PGEN_DIR + "/acaf_threshold.chr%d"

PROJECT_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset"
PROJECTED_OUT = f"{BUCKET_DIR}/round1_pca/acaf_projected"
print(ACAF_PGEN_PATTERN)

### Extract + intersect

Per chromosome: re-ID ACAF to `chr:pos:ref:alt`, extract against `${PRUNE_PREFIX}.prune.in`, merge. Uses the native pgen export (`--pfile`) — the `plink_bed` (PLINK1) export hung indefinitely on this scattered-variant extraction over the network-mounted filesystem; pgen doesn't.

Writes to local scratch disk, not the gcsfuse-mounted bucket directly — 414k samples per chromosome, and large sequential writes over the network-backed mount are much slower than local disk. Only the small final merged result gets copied to the bucket at the end.

In [ ]:
%%bash -s "$ACAF_PGEN_PATTERN" "$PRUNE_PREFIX" "$PROJECT_PREFIX"
set -e

ACAF_PGEN_PATTERN=$1
PRUNE_PREFIX=$2
PROJECT_PREFIX_PATH=$3

PROJECT_PREFIX_NAME=$(basename "$PROJECT_PREFIX_PATH")
BUCKET_OUT_DIR=$(dirname "$PROJECT_PREFIX_PATH")

if [ ! -s "${PRUNE_PREFIX}.prune.in" ]; then
  echo "Missing pruned SNP list: ${PRUNE_PREFIX}.prune.in" >&2
  exit 1
fi

LOCAL_WORK_DIR="$HOME/scratch_acaf"
mkdir -p "$LOCAL_WORK_DIR"
cd "$LOCAL_WORK_DIR"

MERGE_LIST=acaf_merge_list.txt
> "$MERGE_LIST"

for CHR in $(seq 1 22); do
  CHR_PFILE=$(printf "$ACAF_PGEN_PATTERN" "$CHR")

  for EXT in pgen pvar psam; do
    if [ ! -s "${CHR_PFILE}.${EXT}" ]; then
      echo "Missing ACAF input for chr${CHR}: ${CHR_PFILE}.${EXT}" >&2
      exit 1
    fi
  done

  # resumable: skip chromosomes already extracted by a prior (possibly
  # interrupted) run of this cell. Gate on a .done marker written only after
  # plink2 exits 0 -- gating on the .pgen file's mere existence is not
  # enough, since an interrupted run can leave a truncated-but-nonempty .pgen
  # behind that a later --pmerge-list then rejects with "Invalid variant
  # count in .pgen file" (same failure mode as the earlier 0-byte-download bug).
  #
  # Note: doesn't filter multiallelic sites here (--max-alleles 2 would
  # require redoing this extraction against the large network-mounted ACAF
  # source). ACAF genuinely carries extra ALT alleles at some HM3 positions
  # beyond what the reference panel has -- see the post-hoc biallelic filter
  # cell after the merge below, which handles this on the much smaller
  # already-merged output instead.
  if [ ! -s "acaf_chr${CHR}_subset.done" ]; then
    rm -f "acaf_chr${CHR}_subset".{pgen,pvar,psam,log}
    plink2 \
      --pfile "$CHR_PFILE" \
      --set-all-var-ids '@:#:$r:$a' \
      --new-id-max-allele-len 1000 \
      --extract "$PRUNE_PREFIX.prune.in" \
      --make-pgen \
      --out "acaf_chr${CHR}_subset"
    touch "acaf_chr${CHR}_subset.done"
  fi

  echo "acaf_chr${CHR}_subset" >> "$MERGE_LIST"
done

if [ ! -s "${PROJECT_PREFIX_NAME}.done" ]; then
  rm -f "${PROJECT_PREFIX_NAME}".{pgen,pvar,psam,log}
  plink2 --pmerge-list "$MERGE_LIST" pfile --make-pgen --out "$PROJECT_PREFIX_NAME"
  touch "${PROJECT_PREFIX_NAME}.done"
fi

mkdir -p "$BUCKET_OUT_DIR"
cp "${PROJECT_PREFIX_NAME}".pgen "${PROJECT_PREFIX_NAME}".pvar "${PROJECT_PREFIX_NAME}".psam "${PROJECT_PREFIX_NAME}".log "$BUCKET_OUT_DIR/"

N_REF=$(wc -l < "${PRUNE_PREFIX}.prune.in")
N_OVERLAP=$(wc -l < "${PROJECT_PREFIX_NAME}.pvar")
echo "Reference pruned SNPs: $N_REF"
echo "ACAF overlap (incl. pvar header line): $N_OVERLAP"

### Post-hoc biallelic filter + agreeing-SNP list

The per-chromosome extraction above didn't filter multiallelic sites, so `acaf_hm3_subset` can contain ACAF records with extra ALT alleles the reference panel doesn't have (e.g. `REF=T ALT=G,A` in ACAF vs `REF=T ALT=G` in 1000G) -- same ID (built from the first ALT only), but `--read-freq`/`--score` correctly refuse to match them, which is what caused the ~32% "skipped" warnings and the persistent projection misalignment. Rather than re-run the (expensive, network-mounted) per-chromosome extraction with `--max-alleles 2`, filter the already-merged local `acaf_hm3_subset` -- it's small now -- and derive an explicit list of SNPs that agree (ID + REF + ALT) with the reference build. Both projections below use this same list.

In [ ]:
%%bash -s "$PROJECT_PREFIX" "$PCA_PREFIX"
set -e
PROJECT_PREFIX=$1
PCA_PREFIX=$2

cd "$HOME/scratch_acaf"

# 1. filter the already-merged ACAF pgen to biallelic, deduped sites -- reuses
#    the existing extraction, no need to redo the per-chromosome step
BIALLELIC_PREFIX="$(basename "$PROJECT_PREFIX")_biallelic"
plink2 \
  --pfile "$PROJECT_PREFIX" \
  --max-alleles 2 \
  --rm-dup exclude-all \
  --make-pgen \
  --out "$BIALLELIC_PREFIX"

# 2. build the list of SNPs that genuinely agree (ID + REF + ALT) with the
#    reference's own frequency file -- grep -v '^##' strips VCF-style meta
#    lines some ACAF pvar exports carry, which a bare 'NR>1' would otherwise
#    miscount as data rows
grep -v '^##' "${BIALLELIC_PREFIX}.pvar" | awk 'NR>1 {print $3, $4, $5}' | sort > acaf_id_ref_alt.sorted
awk 'NR>1 {print $2, $3, $4}' "${PCA_PREFIX}.acount" | sort > acount_id_ref_alt.sorted
comm -12 acaf_id_ref_alt.sorted acount_id_ref_alt.sorted | awk '{print $1}' > agreeing_snps.ids

echo "Biallelic ACAF variants: $(grep -v '^##' "${BIALLELIC_PREFIX}.pvar" | tail -n +2 | wc -l)"
echo "Agreeing with reference (ID+REF+ALT): $(wc -l < agreeing_snps.ids)"

# 3. copy to the bucket -- local scratch (~/scratch_acaf) isn't guaranteed to
#    survive a session restart, and round 2 (submit_pca_r2.ipynb) needs these
#    two artifacts as persistent inputs, not ephemeral local files
BUCKET_OUT_DIR=$(dirname "$PROJECT_PREFIX")
mkdir -p "$BUCKET_OUT_DIR"
cp "${BIALLELIC_PREFIX}".pgen "${BIALLELIC_PREFIX}".pvar "${BIALLELIC_PREFIX}".psam "${BIALLELIC_PREFIX}".log agreeing_snps.ids "$BUCKET_OUT_DIR/"
echo "Copied ${BIALLELIC_PREFIX}.{pgen,pvar,psam,log} and agreeing_snps.ids to $BUCKET_OUT_DIR/"

### Project

`--score` against the reference's allele weights/counts. Column numbers read off the `.eigenvec.allele` header (layout varies by plink2 build).

In [ ]:
%%bash -s "$PROJECT_PREFIX" "$PCA_PREFIX" "$PROJECTED_OUT"
set -e
PROJECT_PREFIX=$1
PCA_PREFIX=$2
PROJECTED_OUT=$3

cd "$HOME/scratch_acaf"
BIALLELIC_PREFIX="$(basename "$PROJECT_PREFIX")_biallelic"

if [ ! -s agreeing_snps.ids ]; then
  echo "Missing agreeing_snps.ids -- run the post-hoc biallelic filter cell above first" >&2
  exit 1
fi

WEIGHTS="${PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")

ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

# No 'sum' modifier: PC{i}_AVG columns, not PC{i}_SUM. This isn't compared
# against --pca's own eigenvec (a SUM-scale convention) at all -- it's only
# ever compared against the 1000G reprojection below, which uses this same
# AVG scoring, so both sides are on an identical, self-consistent scale
# regardless of which convention that is.
#
# --extract agreeing_snps.ids: restricts to the biallelic-filtered ACAF
# subset that genuinely agrees (ID+REF+ALT) with the reference's own
# frequency file -- see the post-hoc filter cell above for why this was
# necessary (ACAF carries multiallelic sites the reference doesn't).
plink2 \
  --pfile "$BIALLELIC_PREFIX" \
  --extract agreeing_snps.ids \
  --read-freq "${PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$PROJECTED_OUT"

head "${PROJECTED_OUT}.sscore"

### Sanity check: reproject 1000G onto its own PCs

Scoring the reference back onto itself should closely recover the direct `--pca` eigenvectors -- run this before trusting the ACAF projection above. Expect high |correlation| per PC (sign is arbitrary in PCA, so a flipped sign is normal, not a bug); a weak correlation on any PC means something's wrong with the scoring setup itself, independent of any ACAF-specific issues.

Restricted to `agreeing_snps.ids` (from the post-hoc biallelic filter cell) -- the exact SNP set that genuinely agrees between ACAF and the reference -- so this is an apples-to-apples comparison. Run the post-hoc filter and ACAF projection cells above first.

In [ ]:
%%bash -s "$OUT_PREFIX" "$PCA_PREFIX"
set -e
OUT_PREFIX=$1
PCA_PREFIX=$2

cd "$HOME/scratch_acaf"
REPROJECT_OUT="${PCA_PREFIX}_reprojected"

if [ ! -s agreeing_snps.ids ]; then
  echo "Missing agreeing_snps.ids -- run the post-hoc biallelic filter cell above first" >&2
  exit 1
fi

WEIGHTS="${PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")

ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

# No 'sum' modifier -- PC{i}_AVG columns, matching the ACAF projection above,
# so both sides of the round1_filter.ipynb comparison are on the same scale.
plink2 \
  --bfile "$OUT_PREFIX" \
  --extract agreeing_snps.ids \
  --read-freq "${PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$REPROJECT_OUT"

head "${REPROJECT_OUT}.sscore"

In [ ]:
import pandas as pd

direct = pd.read_csv(f"{PCA_PREFIX}.eigenvec", sep=r"\s+")
reproj = pd.read_csv(f"{PCA_PREFIX}_reprojected.sscore", sep=r"\s+")

direct_id_col = "#IID" if "#IID" in direct.columns else "IID"
merged = direct.merge(reproj, left_on=direct_id_col, right_on="IID", suffixes=("_direct", "_reproj"))

score_cols = [c for c in reproj.columns if c.startswith("PC") and c.endswith("_AVG")]
for i, pc in enumerate(range(1, 21)):
    direct_col = f"PC{pc}"
    score_col = score_cols[i] if i < len(score_cols) else None
    if direct_col in merged.columns and score_col in merged.columns:
        corr = merged[direct_col].corr(merged[score_col])
        print(f"PC{pc}: r = {corr:.4f}")